## Build In Tools

In [ ]:
!pip install -U ddgs

In [ ]:
! pip install langchain-experimental

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun ,ShellTool
# from langchain_experimental.tools.python.tool import PythonREPLTool
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)


#tool binding
llm_with_tools = llm.bind_tools([DuckDuckGoSearchRun(), ShellTool()])



In [38]:
res = llm_with_tools.invoke(" use shell to show ls -la output. and then search for 'What is LangChain?' using DuckDuckGoSearchRun tool.")

In [39]:
res

AIMessage(content='', additional_kwargs={'reasoning_content': "We need to run shell command to show ls -la output. Then search for 'What is LangChain?' using DuckDuckGoSearchRun tool. The tool is duckduckgo_search. We need to call terminal first. Then call duckduckgo_search.", 'tool_calls': [{'id': 'fc_f908e912-3b9d-4302-941a-f18bf3db933c', 'function': {'arguments': '{"commands":"ls -la"}', 'name': 'terminal'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 202, 'total_tokens': 280, 'completion_time': 0.079449368, 'prompt_time': 0.009988143, 'queue_time': 0.048893927, 'total_time': 0.089437511, 'completion_tokens_details': {'reasoning_tokens': 53}}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e99e93f2ac', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--365a56f2-e9b7-4b61-b589-50aeb7df5426-0', tool_calls=[{'name': 'terminal', 'args': {'commands': 'l

In [40]:
res.additional_kwargs

{'reasoning_content': "We need to run shell command to show ls -la output. Then search for 'What is LangChain?' using DuckDuckGoSearchRun tool. The tool is duckduckgo_search. We need to call terminal first. Then call duckduckgo_search.",
 'tool_calls': [{'id': 'fc_f908e912-3b9d-4302-941a-f18bf3db933c',
   'function': {'arguments': '{"commands":"ls -la"}', 'name': 'terminal'},
   'type': 'function'}]}

In [30]:
DuckDuckGoSearchRun().invoke(res.tool_calls[0]['args'])

"GDP of France in 2023 - Expenditure Approach, Sector Output, and Regional GDP ... The latest GDP update was in October 2023 , updating the data on the ... ... France 's nominal GDP in 2024 is projected to be around ... Based on PPP, France 's GDP in 2024 is forecast at 4,359 billion international dollars. ... France is expected to grow at a limited pace this year, before recovering in 2024 and 2025, according to Denis Beau, first deputy governor of the ... If we order the countries according to their GDP per capita, France is in 19th position of the 55 countries whose quarterly GDP we publish. The absolute value of GDP in France rose €98,957 $110,247 million with respect to 2023 . ... GDP per capita of France in 2024 was €42,630 $46,184 , ..."

In [41]:
res.tool_calls

[{'name': 'terminal',
  'args': {'commands': 'ls -la'},
  'id': 'fc_f908e912-3b9d-4302-941a-f18bf3db933c',
  'type': 'tool_call'}]

#### Tool execution


In [36]:
ShellTool().invoke(res.tool_calls[0]['args'])

Executing command:
 ['ls -la']


d:\LangChain\.venv\Lib\site-packages\langchain_community\tools\shell\tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


"'ls' is not recognized as an internal or external command,\r\noperable program or batch file.\r\n"

## Custom tools

In [43]:
from langchain_core.tools import tool

@tool
def add(a:float,b:float) -> int:
    '''Adds two numbers together.'''
    return a + b

@tool
def multiply(a:float,b:float) -> float:
    '''Multiplies two numbers together.'''
    return a * b

@tool
def sub(a:float,b:float) -> int:
    '''Subtracts two numbers.'''
    return a - b


In [68]:
from langchain_groq import ChatGroq
model = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

llm_with_custom_tools = model.bind_tools([add, multiply, sub])

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser,JsonOutputParser
prompt = PromptTemplate(
    template="Use the following tools to answer the question below:\n\n {query}"
)

from langchain_core.runnables import RunnableLambda
import json
def extract_tool_calls(ai_msg):
    tool_calls = ai_msg.tool_calls or []
    return json.dumps(tool_calls)      # ✅ convert → string



chain = prompt | llm_with_custom_tools | RunnableLambda(extract_tool_calls)  | JsonOutputParser()

In [72]:
tooltoinvoke = chain.invoke({"query": "What is 10 plus 5 and then multiplied by 3"})

In [73]:
add.invoke(tooltoinvoke[0]['args'])

15.0

In [ ]:

from langchain.agents import create_agent

agent = create_agent(
    model=llm_with_custom_tools,
    tools=[add, multiply, sub],
    system_prompt="You are a helpful assistant",
)

# Run the agent
res = agent.invoke(
    {"messages": [{"role": "user", "content": "add 10 and 20 and then multiply by 3"}]}
)

In [89]:
res['messages']

[HumanMessage(content='add 10 and 20 and then multiply by 3', additional_kwargs={}, response_metadata={}, id='1c9f5af7-80b1-4588-96cd-8d6b017933f9'),
 AIMessage(content='', additional_kwargs={'reasoning_content': "We need to perform add 10 and 20 => 30, then multiply by 3 => 90. Use functions. We'll call add first, then multiply.", 'tool_calls': [{'id': 'fc_9bb8ee8f-09f2-4b24-a97e-da5f80ddc2bd', 'function': {'arguments': '{"a":10,"b":20}', 'name': 'add'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 213, 'total_tokens': 274, 'completion_time': 0.061463452, 'prompt_time': 0.011883684, 'queue_time': 0.054179136, 'total_time': 0.073347136, 'completion_tokens_details': {'reasoning_tokens': 35}}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e99e93f2ac', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--176cafd8-ba22-4a4b-a26d-a1bd5ea5a7e3-0', tool_calls=